# Task 1 MLM

In [1]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

/home/russele7/practicum/dle/sprint_5/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
sentence = "Paris is the capital of [MASK]."
inputs = tokenizer(sentence, return_tensors="pt")
# Используйте позицию [MASK] и получите предсказание
mask_index = torch.where(inputs["input_ids"][0] == tokenizer.mask_token_id)[0].item()

In [4]:
inputs

{'input_ids': tensor([[ 101, 3000, 2003, 1996, 3007, 1997,  103, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [5]:
mask_index

6

In [7]:
# Выполните предсказание и получите логиты
with torch.no_grad(): 
    # Ваш код здесь
    outputs = model(**inputs)
    logits = outputs.logits

In [8]:
outputs

MaskedLMOutput(loss=None, logits=tensor([[[ -6.4449,  -6.4150,  -6.4228,  ...,  -5.7752,  -5.6309,  -3.8315],
         [-11.8119, -12.2862, -12.0384,  ...,  -9.9270, -10.7396,  -7.6424],
         [-10.3959, -10.4091, -10.3610,  ...,  -9.0078,  -6.2265,  -9.0488],
         ...,
         [ -2.9524,  -3.2087,  -2.9790,  ...,  -2.8372,  -5.2569,  -4.8356],
         [-11.6705, -11.4304, -11.7083,  ..., -10.0296, -10.5102,  -6.2767],
         [-14.0257, -14.5389, -14.3912,  ..., -12.9984, -12.2951, -12.2644]]]), hidden_states=None, attentions=None)

In [13]:
logits.shape

torch.Size([1, 9, 30522])

In [16]:
predicted_index = logits[0, mask_index].argmax().item()

In [17]:
predicted_index

2605

In [18]:
print(tokenizer.decode(predicted_index)) 

france


# Task 2 NSP

In [21]:
import torch
from transformers import BertTokenizer, BertForNextSentencePrediction

In [26]:
# Инициализация модели и токенизатора
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForNextSentencePrediction.from_pretrained('bert-base-uncased')

In [ ]:
# Пример для IsNext (последовательные предложения)
text_a = 'The cat sat on the mat'  # Ваш код здесь
text_b = 'It was very sleepy'  # Ваш код здесь

In [28]:
# Токенизация и подготовка входа
inputs = tokenizer(text_a, text_b, return_tensors='pt')  # Ваш код здесь

In [29]:
# Получение предсказаний
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [34]:
torch.softmax(logits, dim=1)[0]

tensor([9.9994e-01, 6.2259e-05])

In [30]:
# Интерпретация результатов
is_next_prob = torch.softmax(logits, dim=1)[0][1].item()  # Вероятность NotNext
not_next_prob = torch.softmax(logits, dim=1)[0][0].item() # Вероятность IsNext

In [31]:
print(f"Вероятность IsNext: {not_next_prob:.4f}")
print(f"Вероятность NotNext: {is_next_prob:.4f}")
print("\n" + "="*80 + "\n") 

Вероятность IsNext: 0.9999
Вероятность NotNext: 0.0001




# Task 3 NSP с новым примером

In [35]:
# Пример для NotNext (не последовательные предложения)
text_a = 'The cat sat on the mat' # Ваш код здесь
text_b = 'The Eiffel Tower is located in Paris' # Ваш код здесь

In [36]:
# Токенизация и подготовка входа
inputs = tokenizer(text_a, text_b, return_tensors='pt')

# Получение предсказаний
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [39]:
inputs['input_ids'].shape

torch.Size([1, 18])

In [40]:
logits

tensor([[-2.7603,  5.9250]])

In [41]:
# Интерпретация результатов
is_next_prob = torch.softmax(logits, dim=1)[0][1].item()  # Вероятность NotNext
not_next_prob = torch.softmax(logits, dim=1)[0][0].item() # Вероятность IsNext

In [42]:
print(f"Вероятность IsNext: {not_next_prob:.4f}")
print(f"Вероятность NotNext: {is_next_prob:.4f}")
print("\n" + "="*80 + "\n")

Вероятность IsNext: 0.0002
Вероятность NotNext: 0.9998




# Задание 4. Анализ потребления памяти

In [43]:
torch.cuda.is_available()

True

In [44]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained('bert-base-uncased').cuda()

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 45011021-9635-4048-a73f-1618929041cf)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
# Проверка работы GPU
# Ваш код здесь

for seq_len in  [128, 256, 512]: # Ваш код здесь
    x = torch.randint(0, model.config.vocab_size, (1, seq_len)).cuda()
    torch.cuda.reset_peak_memory_stats()
    _ = model(x)
    print(f"Длина {seq_len}: пиковое потребление ≈ {torch.cuda.max_memory_allocated()/1024**2:.0f} МБ") 

Длина 128: пиковое потребление ≈ 510 МБ
Длина 256: пиковое потребление ≈ 657 МБ
Длина 512: пиковое потребление ≈ 884 МБ
